In [1]:
import qi
import os
import time
import threading

from naoqi import ALProxy


# Bot's IP address and port
ip = "192.168.0.192"
port = 9559

In [2]:
file_manager = ALProxy("ALFileManager", ip, port)
shared_folder_name = file_manager.getUserSharedFolderPath()
shared_folder_name

'/home/nao/'

In [ ]:
# Record Video
def record_video(timer=10):
    video_recorder = ALProxy("ALVideoRecorder", ip, port)
    video_recorder.startRecording("/home/nao/recordings/cameras", "test_video")
    time.sleep(timer)
    videoInfo = video_recorder.stopRecording()

    print('Video was saved on the robot: ', videoInfo[1])
    print('Total number of frames: ', videoInfo[0])

('Video was saved on the robot: ', '/home/nao/recordings/cameras/test_video.avi')
('Total number of frames: ', 73)


In [4]:
# Record the audio
def record_audio(timer=10, path_name="/home/nao/microphones/recording.wav"):
    path_name = os.path.join(path_name)
    # Create a proxy to ALAudioRecorder
    recorder = ALProxy("ALAudioRecorder", ip, port)
    # Start recording
    recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
    time.sleep(timer)
    # Stop recording
    recorder.stopMicrophonesRecording()

In [27]:
record_audio(5)

**Method 1**

In [24]:
!pip install paramiko
import paramiko
def transfer_file(remote_path="/home/nao/microphones/recording.wav", local_path="pepper-bot/recordings/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.get(remote_path, local_path)

    sftp.close()
    ssh.close()

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.


In [28]:
transfer_file()

**Method 2**

In [ ]:
!pip install pyftplib
import pyftplib
from pyftplib import FTPServer
from pyftplib.handlers import FTPHandler
from pyftplib.authorizers import DummyAuthorizer
import threading

def ftp_transfer_file():
    remote_path = "/home/nao/microphones/recording.wav"
    authorizer = DummyAuthorizer()
    authorizer.add_user("nao", "nao", remote_path, perm="elradfmw")
    handler = FTPHandler
    handler.authorizer = authorizer
    server = FTPServer((ip, 21), handler)
    server_thread = threading.Thread(target=server.serve_forever)
    server_thread.daemon = True
    server_thread.start()

# Tablet

In [23]:
tablet_service = ALProxy("ALTabletService", ip, port)
tablet_service.wakeUp()

In [ ]:
!pip install pyftpdlib